**A & B. Streaming Source & Branch Aggregation**

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
import pandas as pd
import os
import shutil
import time
from datetime import datetime

spark = SparkSession.builder.appName("Tugas10_Dany_Akhdan_Imaduddin").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)
print(f"Simulation folder '{folder_tugas}' is ready.")

df_stream = spark.readStream.format("csv").schema(skema_pesanan).option("header", True).load(folder_tugas)
df_stream = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

# B. Aggregation per Branch
agg_cabang = df_stream.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("pendapatan_total"),
    count("pesanan_id").alias("jumlah_pesanan")
)
query_cabang = agg_cabang.writeStream.outputMode("complete").format("memory").queryName("agregasi_cabang").start()
print("Streaming query for branch aggregation started successfully!")
print("Query Status:", query_cabang.isActive)

Simulation folder 'tugas10_stream' is ready.
Streaming query for branch aggregation started successfully!
Query Status: True


**C. Time Window-Based Aggregation**

In [6]:
agg_waktu = df_stream.groupBy(window(col("waktu_pesan"), "2 minute")).agg(
    spark_sum("total_harga").alias("pendapatan_total")
)
query_waktu = agg_waktu.writeStream.outputMode("complete").format("memory").queryName("agregasi_waktu").start()
print("Streaming query for 2-minute time window aggregation started successfully!")
print("Query Status:", query_waktu.isActive)

Streaming query for 2-minute time window aggregation started successfully!
Query Status: True


**D. Testing with Simulated Data (3 Batches)**

In [8]:
waktu_sekarang = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# BATCH 1
pd.DataFrame({
    "pesanan_id": ["P1", "P2"], "nama_cabang": ["Semarang", "Yogyakarta"],
    "nama_menu": ["Americano", "Latte"], "qty": [2, 1], "harga_satuan": [20000, 25000], "waktu_pesan": [waktu_sekarang, waktu_sekarang]
}).to_csv(f"{folder_tugas}/batch_1.csv", index=False)
print("Batch 1 sent... Waiting for Spark to process...")
time.sleep(6)
print("Current Results for Branch Aggregation (After Batch 1):")
spark.sql("SELECT * FROM agregasi_cabang").show()

# BATCH 2
pd.DataFrame({
    "pesanan_id": ["P3", "P4"], "nama_cabang": ["Semarang", "Magelang"],
    "nama_menu": ["Matcha", "Espresso"], "qty": [1, 3], "harga_satuan": [27000, 18000], "waktu_pesan": [waktu_sekarang, waktu_sekarang]
}).to_csv(f"{folder_tugas}/batch_2.csv", index=False)
print("Batch 2 sent... Waiting for Spark to process...")
time.sleep(6)
print("Current Results for Branch Aggregation (After Batch 2):")
spark.sql("SELECT * FROM agregasi_cabang").show()

# BATCH 3
pd.DataFrame({
    "pesanan_id": ["P5"], "nama_cabang": ["Yogyakarta"],
    "nama_menu": ["Nasi Goreng"], "qty": [2], "harga_satuan": [30000], "waktu_pesan": [waktu_sekarang]
}).to_csv(f"{folder_tugas}/batch_3.csv", index=False)
print("Batch 3 sent... Waiting for Spark to process...")
time.sleep(6)
print("Current Results for Branch Aggregation (After Batch 3):")
spark.sql("SELECT * FROM agregasi_cabang").show()

Batch 1 sent... Waiting for Spark to process...
Current Results for Branch Aggregation (After Batch 1):
+-----------+----------------+--------------+
|nama_cabang|pendapatan_total|jumlah_pesanan|
+-----------+----------------+--------------+
|   Semarang|           67000|             2|
| Yogyakarta|           85000|             2|
|   Magelang|           54000|             1|
+-----------+----------------+--------------+

Batch 2 sent... Waiting for Spark to process...
Current Results for Branch Aggregation (After Batch 2):
+-----------+----------------+--------------+
|nama_cabang|pendapatan_total|jumlah_pesanan|
+-----------+----------------+--------------+
|   Semarang|           67000|             2|
| Yogyakarta|           85000|             2|
|   Magelang|           54000|             1|
+-----------+----------------+--------------+

Batch 3 sent... Waiting for Spark to process...
Current Results for Branch Aggregation (After Batch 3):
+-----------+----------------+------------

**E. File-Based Sink (Challenge)**

In [9]:
folder_output = "tugas10_output"
folder_checkpoint = "tugas10_checkpoint"
shutil.rmtree(folder_output, ignore_errors=True)
shutil.rmtree(folder_checkpoint, ignore_errors=True)

query_file = df_stream.writeStream \
    .format("parquet") \
    .option("path", folder_output) \
    .option("checkpointLocation", folder_checkpoint) \
    .outputMode("append") \
    .start()
print("Streaming query for Parquet file sink started successfully!")
print("Query Status:", query_file.isActive)

# Send an extra batch specifically for the Parquet Sink
pd.DataFrame({
    "pesanan_id": ["P6"], "nama_cabang": ["Jakarta"],
    "nama_menu": ["Es Teh"], "qty": [5], "harga_satuan": [10000], "waktu_pesan": [waktu_sekarang]
}).to_csv(f"{folder_tugas}/batch_parquet.csv", index=False)
print("Extra batch sent for Parquet sink... Processing...")

time.sleep(6)
print("Reading back the Raw Parquet data from the Output folder:")
spark.read.parquet(folder_output).show()

# Stopping all queries safely
query_cabang.stop()
query_waktu.stop()
query_file.stop()
print("All assignment streaming queries have been safely stopped.")

Streaming query for Parquet file sink started successfully!
Query Status: True
Extra batch sent for Parquet sink... Processing...


Reading back the Raw Parquet data from the Output folder:
+----------+-----------+-----------+---+------------+-------------------+-----------+
|pesanan_id|nama_cabang|  nama_menu|qty|harga_satuan|        waktu_pesan|total_harga|
+----------+-----------+-----------+---+------------+-------------------+-----------+
|        P5| Yogyakarta|Nasi Goreng|  2|       30000|2026-11-11 23:18:30|      60000|
|        P1|   Semarang|  Americano|  2|       20000|2026-11-11 23:18:30|      40000|
|        P2| Yogyakarta|      Latte|  1|       25000|2026-11-11 23:18:30|      25000|
|        P3|   Semarang|     Matcha|  1|       27000|2026-11-11 23:18:30|      27000|
|        P4|   Magelang|   Espresso|  3|       18000|2026-11-11 23:18:30|      54000|
|        P6|    Jakarta|     Es Teh|  5|       10000|2026-11-11 23:18:30|      50000|
+----------+-----------+-----------+---+------------+-------------------+-----------+

All assignment streaming queries have been safely stopped.


In [10]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
